# Hackathon Judge SFT — Kaggle Smoke Test
Small model, low rank, tiny dataset. Runs end-to-end in ~10 min on T4.

In [ ]:
# ── params ────────────────────────────────────────────────
MODEL      = "unsloth/Qwen3-0.6B"
HACKATHON  = "madhacks"   # smallest config (~460 trainable rows)
N_TRAIN    = 40           # pairs to train on  (80 rows with ab+ba)
N_EVAL     = 10           # pairs to evaluate
LORA_R     = 8
EPOCHS     = 1
OUTPUT_DIR = "/kaggle/working/lora_test"
# ──────────────────────────────────────────────────────────

In [ ]:
%%capture
!pip install "unsloth[kaggle-new]" trl "datasets>=4.8.5" pyarrow peft

In [ ]:
import random, re, json, time
from datasets import load_dataset

ds = load_dataset("twangodev/devpost-hacks-judgments", HACKATHON)["train"]
ds = ds.filter(lambda r: r["model"] == "Qwen/Qwen3.5-27B" and r["verdict"] in ("A", "B"))

# split by pair_id
pairs = list(set(ds["pair_id"]))
random.seed(42)
random.shuffle(pairs)
train_pairs = set(pairs[:N_TRAIN])
eval_pairs  = set(pairs[N_TRAIN:N_TRAIN + N_EVAL])

train_ds = ds.filter(lambda r: r["pair_id"] in train_pairs)
eval_ds  = ds.filter(lambda r: r["pair_id"] in eval_pairs)
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=8192,
    load_in_4bit=True,
    full_finetuning=False,
)
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing="unsloth",
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
from trl import SFTTrainer, SFTConfig

def preprocess(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

train_tokenized = train_ds.map(preprocess, num_proc=1)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_tokenized,
    max_seq_length=8192,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=5,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataset_text_field="text",
    ),
)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"saved → {OUTPUT_DIR}")

In [ ]:
from peft import PeftModel

base, tokenizer = FastModel.from_pretrained(MODEL, max_seq_length=8192, load_in_4bit=True)
eval_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
eval_model.eval()

VERDICT_RE = re.compile(r"VERDICT\s*:\s*(A|B|TIE)\b", re.IGNORECASE)

def parse_verdict(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    last = None
    for m in VERDICT_RE.finditer(text):
        last = m
    if last is None:
        return None
    v = last.group(1).upper()
    return "tie" if v == "TIE" else v

def infer(messages):
    prompt = tokenizer.apply_chat_template(
        [m for m in messages if m["role"] != "assistant"],
        tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=2048, temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
results = []
for ex in eval_ds:
    response = infer(ex["messages"])
    predicted = parse_verdict(response)
    match = predicted == ex["verdict"]
    results.append({"pair_id": ex["pair_id"], "position": ex["position"],
                    "predicted": predicted, "frontier": ex["verdict"], "match": match})
    print(f"{ex['pair_id'][:8]} pos={ex['position']}  predicted={predicted}  frontier={ex['verdict']}  {'✓' if match else '✗'}")

agreement = sum(r["match"] for r in results) / len(results)

pair_map = {}
for r in results:
    pair_map.setdefault(r["pair_id"], {})[r["position"]] = r["predicted"]
consistent = sum(
    1 for v in pair_map.values()
    if (v.get("ab") == "A" and v.get("ba") == "B") or (v.get("ab") == "B" and v.get("ba") == "A")
)
n_pairs = sum(1 for v in pair_map.values() if "ab" in v and "ba" in v)

print(f"\nfrontier agreement:   {agreement*100:.1f}% ({len(results)} examples)")
print(f"position consistency: {consistent/n_pairs*100:.0f}% ({n_pairs} pairs)")